# MLP Backpropagation: From Scratch

This notebook re-derives backpropagation for a Multi-Layer Perceptron (MLP) and implements a 2-3 layer MLP from scratch using NumPy.

## Goals
1. Derive backpropagation mathematically
2. Implement MLP from scratch (no deep learning frameworks)
3. Train on a toy dataset
4. Visualize activations and gradients


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style (try different style names for compatibility)
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('default')
sns.set_palette("husl")


## 1. Mathematical Derivation of Backpropagation

### Forward Pass

For an MLP with L layers:

**Layer l:**
- Input: $a^{(l-1)}$ (activation from previous layer)
- Linear transformation: $z^{(l)} = W^{(l)} a^{(l-1)} + b^{(l)}$
- Activation: $a^{(l)} = \sigma(z^{(l)})$

Where:
- $W^{(l)}$ is the weight matrix of shape $(n_l, n_{l-1})$
- $b^{(l)}$ is the bias vector of shape $(n_l,)$
- $\sigma$ is the activation function (e.g., ReLU, sigmoid, tanh)

### Loss Function

For a single sample $(x, y)$:
- Forward pass: $a^{(0)} = x$, then compute $a^{(1)}, ..., a^{(L)}$
- Loss: $L = \mathcal{L}(a^{(L)}, y)$ (e.g., MSE for regression, cross-entropy for classification)

### Backward Pass (Backpropagation)

We need to compute gradients: $\frac{\partial L}{\partial W^{(l)}}$ and $\frac{\partial L}{\partial b^{(l)}}$

**Key insight:** Use chain rule starting from the output layer.

**Output layer (L):**
- $\delta^{(L)} = \frac{\partial L}{\partial a^{(L)}} \odot \sigma'(z^{(L)})$
- $\frac{\partial L}{\partial W^{(L)}} = \delta^{(L)} (a^{(L-1)})^T$
- $\frac{\partial L}{\partial b^{(L)}} = \delta^{(L)}$

**Hidden layers (l = L-1, ..., 1):**
- $\delta^{(l)} = ((W^{(l+1)})^T \delta^{(l+1)}) \odot \sigma'(z^{(l)})$
- $\frac{\partial L}{\partial W^{(l)}} = \delta^{(l)} (a^{(l-1)})^T$
- $\frac{\partial L}{\partial b^{(l)}} = \delta^{(l)}$

Where $\odot$ denotes element-wise multiplication.


## 2. Implementation: MLP from Scratch


In [ ]:
class MLP:
    """
    Multi-Layer Perceptron implemented from scratch using NumPy.
    Supports 2-3 layers with configurable activation functions.
    """
    
    def __init__(self, layer_sizes, activation='relu', output_activation='sigmoid', 
                 initialization='xavier'):
        """
        Initialize MLP.
        
        Args:
            layer_sizes: List of layer sizes, e.g., [input_dim, hidden1, hidden2, output_dim]
            activation: Activation function for hidden layers ('relu', 'tanh', 'sigmoid')
            output_activation: Activation function for output layer
            initialization: Weight initialization method ('xavier', 'he', 'random')
        """
        self.layer_sizes = layer_sizes
        self.num_layers = len(layer_sizes) - 1
        self.activation = activation
        self.output_activation = output_activation
        
        # Initialize weights and biases
        self.weights = []
        self.biases = []
        
        for i in range(self.num_layers):
            n_in = layer_sizes[i]
            n_out = layer_sizes[i + 1]
            
            if initialization == 'xavier':
                # Xavier/Glorot initialization
                limit = np.sqrt(6.0 / (n_in + n_out))
                W = np.random.uniform(-limit, limit, (n_out, n_in))
            elif initialization == 'he':
                # He initialization (good for ReLU)
                std = np.sqrt(2.0 / n_in)
                W = np.random.randn(n_out, n_in) * std
            else:
                # Random initialization
                W = np.random.randn(n_out, n_in) * 0.1
            
            b = np.zeros((n_out, 1))
            
            self.weights.append(W)
            self.biases.append(b)
        
        # Storage for forward pass (for backprop)
        self.z_values = []
        self.activations = []
        
        # Storage for gradients (for visualization)
        self.gradients = {'weights': [], 'biases': []}
    
    def _activation_fn(self, z, activation_type):
        """Apply activation function."""
        if activation_type == 'relu':
            return np.maximum(0, z)
        elif activation_type == 'tanh':
            return np.tanh(z)
        elif activation_type == 'sigmoid':
            return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # Clip to avoid overflow
        else:
            raise ValueError(f"Unknown activation: {activation_type}")
    
    def _activation_derivative(self, z, activation_type):
        """Compute derivative of activation function."""
        if activation_type == 'relu':
            return (z > 0).astype(float)
        elif activation_type == 'tanh':
            return 1 - np.tanh(z)**2
        elif activation_type == 'sigmoid':
            s = self._activation_fn(z, 'sigmoid')
            return s * (1 - s)
        else:
            raise ValueError(f"Unknown activation: {activation_type}")
    
    def forward(self, X):
        """
        Forward pass through the network.
        
        Args:
            X: Input data of shape (n_features, n_samples)
        
        Returns:
            Output of the network
        """
        # Reset storage
        self.z_values = []
        self.activations = [X]  # Store input as a^(0)
        
        a = X
        for i in range(self.num_layers):
            # Linear transformation
            z = self.weights[i] @ a + self.biases[i]
            self.z_values.append(z)
            
            # Apply activation
            if i == self.num_layers - 1:
                # Output layer
                a = self._activation_fn(z, self.output_activation)
            else:
                # Hidden layers
                a = self._activation_fn(z, self.activation)
            
            self.activations.append(a)
        
        return a
    
    def backward(self, X, y, output):
        """
        Backward pass (backpropagation).
        
        Args:
            X: Input data of shape (n_features, n_samples)
            y: True labels of shape (n_output, n_samples)
            output: Network output from forward pass
        
        Returns:
            Dictionary with gradients for weights and biases
        """
        m = X.shape[1]  # Number of samples
        
        # Initialize gradients
        dW = [np.zeros_like(W) for W in self.weights]
        db = [np.zeros_like(b) for b in self.biases]
        
        # Output layer error
        # For binary classification with sigmoid + BCE loss:
        # dL/da^(L) = (a^(L) - y) / (a^(L) * (1 - a^(L))) * a^(L) * (1 - a^(L)) = a^(L) - y
        delta = output - y
        
        # Backpropagate through layers
        for l in range(self.num_layers - 1, -1, -1):
            # Gradient w.r.t. weights and biases
            dW[l] = (1 / m) * delta @ self.activations[l].T
            db[l] = (1 / m) * np.sum(delta, axis=1, keepdims=True)
            
            # Propagate error to previous layer (if not input layer)
            if l > 0:
                # Error for previous layer
                delta = (self.weights[l].T @ delta) * \
                        self._activation_derivative(self.z_values[l-1], self.activation)
        
        # Store gradients for visualization
        self.gradients['weights'] = [dw.copy() for dw in dW]
        self.gradients['biases'] = [db.copy() for db in db]
        
        return {'weights': dW, 'biases': db}
    
    def update_weights(self, gradients, learning_rate):
        """Update weights using gradients."""
        for i in range(self.num_layers):
            self.weights[i] -= learning_rate * gradients['weights'][i]
            self.biases[i] -= learning_rate * gradients['biases'][i]
    
    def compute_loss(self, y_true, y_pred):
        """Compute binary cross-entropy loss."""
        # Clip predictions to avoid log(0)
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
        return loss
    
    def predict(self, X):
        """Make predictions."""
        output = self.forward(X)
        return (output > 0.5).astype(int)
    
    def get_activations(self):
        """Get activations from last forward pass."""
        return self.activations
    
    def get_gradients(self):
        """Get gradients from last backward pass."""
        return self.gradients


## 3. Generate Toy Dataset


In [ ]:
# Generate a toy dataset (moons dataset for binary classification)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

# Visualize the dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c='blue', label='Class 0', alpha=0.6)
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='red', label='Class 1', alpha=0.6)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Toy Dataset: Two Moons')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Reshape for our MLP (features x samples)
X_train = X_train.T
X_test = X_test.T
y_train = y_train.reshape(1, -1)
y_test = y_test.reshape(1, -1)

print(f"Training set: {X_train.shape[1]} samples")
print(f"Test set: {X_test.shape[1]} samples")


## 4. Train the MLP


In [ ]:
# Initialize MLP: 2 input features -> 10 hidden -> 5 hidden -> 1 output
mlp = MLP(layer_sizes=[2, 10, 5, 1], 
          activation='relu', 
          output_activation='sigmoid',
          initialization='he')

# Training parameters
learning_rate = 0.01
num_epochs = 1000
batch_size = 32

# Storage for training history
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []

# Storage for activations and gradients (we'll sample periodically)
activation_history = []
gradient_history = []

print("Training MLP...")
for epoch in range(num_epochs):
    # Shuffle training data
    indices = np.random.permutation(X_train.shape[1])
    X_train_shuffled = X_train[:, indices]
    y_train_shuffled = y_train[:, indices]
    
    epoch_loss = 0
    num_batches = 0
    
    # Mini-batch training
    for i in range(0, X_train.shape[1], batch_size):
        X_batch = X_train_shuffled[:, i:i+batch_size]
        y_batch = y_train_shuffled[:, i:i+batch_size]
        
        # Forward pass
        output = mlp.forward(X_batch)
        
        # Compute loss
        loss = mlp.compute_loss(y_batch, output)
        epoch_loss += loss
        num_batches += 1
        
        # Backward pass
        gradients = mlp.backward(X_batch, y_batch, output)
        
        # Update weights
        mlp.update_weights(gradients, learning_rate)
    
    # Average loss for epoch
    avg_loss = epoch_loss / num_batches
    train_losses.append(avg_loss)
    
    # Evaluate on test set
    test_output = mlp.forward(X_test)
    test_loss = mlp.compute_loss(y_test, test_output)
    test_losses.append(test_loss)
    
    # Compute accuracies
    train_pred = mlp.predict(X_train)
    test_pred = mlp.predict(X_test)
    train_acc = np.mean(train_pred == y_train)
    test_acc = np.mean(test_pred == y_test)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    # Store activations and gradients periodically
    if epoch % 100 == 0 or epoch < 10:
        activation_history.append({
            'epoch': epoch,
            'activations': [a.copy() for a in mlp.get_activations()],
            'z_values': [z.copy() for z in mlp.z_values]
        })
        gradient_history.append({
            'epoch': epoch,
            'weight_grads': [g.copy() for g in mlp.gradients['weights']],
            'bias_grads': [g.copy() for g in mlp.gradients['biases']]
        })
    
    # Print progress
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_loss:.4f}, "
              f"Test Loss: {test_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

print("\nTraining complete!")


## 5. Plot Training Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Train Loss', linewidth=2)
axes[0].plot(test_losses, label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Test Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(train_accuracies, label='Train Accuracy', linewidth=2)
axes[1].plot(test_accuracies, label='Test Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Test Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 6. Visualize Decision Boundary


In [ ]:
# Create a mesh to plot decision boundary
h = 0.02
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predict on mesh
mesh_points = np.c_[xx.ravel(), yy.ravel()].T
Z = mlp.forward(mesh_points)
Z = Z.reshape(xx.shape)

# Plot
plt.figure(figsize=(10, 8))
plt.contourf(xx, yy, Z, levels=50, alpha=0.6, cmap='RdYlBu')
plt.colorbar(label='Predicted Probability')
plt.scatter(X[y == 0, 0], X[y == 0, 1], c='blue', label='Class 0', 
            edgecolors='black', linewidths=0.5, s=50)
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='red', label='Class 1', 
            edgecolors='black', linewidths=0.5, s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('MLP Decision Boundary')
plt.legend()
plt.tight_layout()
plt.show()


## 7. Visualize Activations


In [ ]:
# Select a few samples to visualize
num_samples_to_plot = 5
sample_indices = np.random.choice(X_test.shape[1], num_samples_to_plot, replace=False)

# Forward pass on selected samples
X_samples = X_test[:, sample_indices]
mlp.forward(X_samples)
activations = mlp.get_activations()

# Plot activations for each layer
fig, axes = plt.subplots(len(activations), num_samples_to_plot, 
                         figsize=(15, 3 * len(activations)))

if num_samples_to_plot == 1:
    axes = axes.reshape(-1, 1)

for layer_idx, layer_activations in enumerate(activations):
    for sample_idx in range(num_samples_to_plot):
        ax = axes[layer_idx, sample_idx]
        
        if layer_idx == 0:
            # Input layer - show as 2D scatter or bar
            ax.bar(range(layer_activations.shape[0]), 
                   layer_activations[:, sample_idx], 
                   color='skyblue', edgecolor='black')
            ax.set_title(f'Sample {sample_idx+1}\nInput Layer')
        else:
            # Hidden/output layers - show as bar chart
            ax.bar(range(layer_activations.shape[0]), 
                   layer_activations[:, sample_idx], 
                   color='lightcoral', edgecolor='black')
            ax.set_title(f'Layer {layer_idx}')
        
        ax.set_ylabel('Activation')
        ax.set_xlabel('Neuron')
        ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Activations Across Layers for Different Samples', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 8. Visualize Activation Statistics Over Training


In [ ]:
# Compute activation statistics for each layer over training
if len(activation_history) > 0:
    num_layers = len(activation_history[0]['activations']) - 1  # Exclude input
    
    fig, axes = plt.subplots(num_layers, 2, figsize=(14, 4 * num_layers))
    if num_layers == 1:
        axes = axes.reshape(1, -1)
    
    epochs = [h['epoch'] for h in activation_history]
    
    for layer_idx in range(num_layers):
        # Mean activations
        mean_activations = []
        std_activations = []
        
        for hist in activation_history:
            layer_acts = hist['activations'][layer_idx + 1]  # +1 because activations[0] is input
            mean_activations.append(np.mean(layer_acts))
            std_activations.append(np.std(layer_acts))
        
        axes[layer_idx, 0].plot(epochs, mean_activations, 'o-', linewidth=2, label='Mean')
        axes[layer_idx, 0].fill_between(epochs, 
                                        np.array(mean_activations) - np.array(std_activations),
                                        np.array(mean_activations) + np.array(std_activations),
                                        alpha=0.3, label='±1 Std')
        axes[layer_idx, 0].set_xlabel('Epoch')
        axes[layer_idx, 0].set_ylabel('Activation Value')
        axes[layer_idx, 0].set_title(f'Layer {layer_idx + 1} Activation Statistics')
        axes[layer_idx, 0].legend()
        axes[layer_idx, 0].grid(True, alpha=0.3)
        
        # Distribution of activations at different epochs
        selected_epochs = [0, len(activation_history)//2, len(activation_history)-1]
        for epoch_idx in selected_epochs:
            if epoch_idx < len(activation_history):
                hist = activation_history[epoch_idx]
                layer_acts = hist['activations'][layer_idx + 1]
                axes[layer_idx, 1].hist(layer_acts.flatten(), 
                                       bins=30, alpha=0.5, 
                                       label=f"Epoch {hist['epoch']}")
        
        axes[layer_idx, 1].set_xlabel('Activation Value')
        axes[layer_idx, 1].set_ylabel('Frequency')
        axes[layer_idx, 1].set_title(f'Layer {layer_idx + 1} Activation Distribution')
        axes[layer_idx, 1].legend()
        axes[layer_idx, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()


## 9. Visualize Gradients


In [ ]:
# Visualize gradient magnitudes over training
if len(gradient_history) > 0:
    num_layers = len(gradient_history[0]['weight_grads'])
    
    fig, axes = plt.subplots(num_layers, 2, figsize=(14, 4 * num_layers))
    if num_layers == 1:
        axes = axes.reshape(1, -1)
    
    epochs = [h['epoch'] for h in gradient_history]
    
    for layer_idx in range(num_layers):
        # Gradient magnitudes (L2 norm)
        weight_grad_norms = []
        bias_grad_norms = []
        
        for hist in gradient_history:
            w_grad = hist['weight_grads'][layer_idx]
            b_grad = hist['bias_grads'][layer_idx]
            weight_grad_norms.append(np.linalg.norm(w_grad))
            bias_grad_norms.append(np.linalg.norm(b_grad))
        
        axes[layer_idx, 0].plot(epochs, weight_grad_norms, 'o-', 
                               linewidth=2, label='Weight Gradients', color='blue')
        axes[layer_idx, 0].plot(epochs, bias_grad_norms, 's-', 
                               linewidth=2, label='Bias Gradients', color='red')
        axes[layer_idx, 0].set_xlabel('Epoch')
        axes[layer_idx, 0].set_ylabel('Gradient Norm (L2)')
        axes[layer_idx, 0].set_title(f'Layer {layer_idx + 1} Gradient Magnitudes')
        axes[layer_idx, 0].set_yscale('log')
        axes[layer_idx, 0].legend()
        axes[layer_idx, 0].grid(True, alpha=0.3)
        
        # Gradient distribution at different epochs
        selected_epochs = [0, len(gradient_history)//2, len(gradient_history)-1]
        for epoch_idx in selected_epochs:
            if epoch_idx < len(gradient_history):
                hist = gradient_history[epoch_idx]
                w_grad = hist['weight_grads'][layer_idx]
                axes[layer_idx, 1].hist(w_grad.flatten(), bins=30, alpha=0.5,
                                       label=f"Epoch {hist['epoch']}")
        
        axes[layer_idx, 1].set_xlabel('Gradient Value')
        axes[layer_idx, 1].set_ylabel('Frequency')
        axes[layer_idx, 1].set_title(f'Layer {layer_idx + 1} Weight Gradient Distribution')
        axes[layer_idx, 1].legend()
        axes[layer_idx, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()


## 10. Gradient Flow Analysis

Let's visualize how gradients flow through the network by plotting gradient magnitudes at each layer.


In [ ]:
# Analyze gradient flow through layers
if len(gradient_history) > 0:
    # Get gradient norms for each layer at different training stages
    early_epoch = 0
    mid_epoch = len(gradient_history) // 2
    late_epoch = len(gradient_history) - 1
    
    epochs_to_plot = [early_epoch, mid_epoch, late_epoch]
    epoch_labels = ['Early Training', 'Mid Training', 'Late Training']
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x_pos = np.arange(num_layers)
    width = 0.25
    
    for i, epoch_idx in enumerate(epochs_to_plot):
        if epoch_idx < len(gradient_history):
            hist = gradient_history[epoch_idx]
            grad_norms = [np.linalg.norm(g) for g in hist['weight_grads']]
            ax.bar(x_pos + i * width, grad_norms, width, 
                  label=epoch_labels[i], alpha=0.8)
    
    ax.set_xlabel('Layer')
    ax.set_ylabel('Gradient Norm (L2)')
    ax.set_title('Gradient Flow Through Layers')
    ax.set_xticks(x_pos + width)
    ax.set_xticklabels([f'Layer {i+1}' for i in range(num_layers)])
    ax.set_yscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
    
    # Check for vanishing/exploding gradients
    print("\nGradient Analysis:")
    print("-" * 50)
    for epoch_idx in epochs_to_plot:
        if epoch_idx < len(gradient_history):
            hist = gradient_history[epoch_idx]
            grad_norms = [np.linalg.norm(g) for g in hist['weight_grads']]
            print(f"\n{epoch_labels[epochs_to_plot.index(epoch_idx)]} (Epoch {hist['epoch']}):")
            for layer_idx, norm in enumerate(grad_norms):
                status = ""
                if norm < 1e-6:
                    status = " (Vanishing!)"
                elif norm > 1e2:
                    status = " (Exploding!)"
                print(f"  Layer {layer_idx + 1}: {norm:.6e}{status}")


## Summary

This notebook demonstrated:

1. **Mathematical Derivation**: We derived the backpropagation algorithm step-by-step using the chain rule.

2. **Implementation from Scratch**: We built a complete MLP using only NumPy, including:
   - Forward pass with configurable activation functions
   - Backward pass (backpropagation) with gradient computation
   - Weight initialization methods (Xavier, He)
   - Training loop with mini-batch gradient descent

3. **Training**: We trained the MLP on a toy dataset (two moons) and achieved good classification performance.

4. **Visualizations**:
   - Training curves (loss and accuracy)
   - Decision boundary
   - Activations across layers for different samples
   - Activation statistics over training
   - Gradient magnitudes and distributions
   - Gradient flow analysis

Key insights:
- Activations show how information flows forward through the network
- Gradients show how error signals flow backward during training
- Monitoring gradients helps detect vanishing/exploding gradient problems
- The network learns to separate the two classes effectively
